# load dataset

In [1]:
import pandas as pd
dataset = pd.read_csv("processed_ai4i2020.csv")
dataset.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Temp_diff [K]
0,2,298.1,308.6,1551,42.8,0,0,10.5
1,1,298.2,308.7,1408,46.3,3,0,10.5
2,1,298.1,308.5,1498,49.4,5,0,10.4
3,1,298.2,308.6,1433,39.5,7,0,10.4
4,1,298.2,308.7,1408,40.0,9,0,10.5


In [2]:
dataset.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'Temp_diff [K]'],
      dtype='object')

# input output split

In [4]:
x=dataset.drop(['Machine failure'],axis=1)
y=dataset['Machine failure']

In [5]:
x.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Temp_diff [K]
0,2,298.1,308.6,1551,42.8,0,10.5
1,1,298.2,308.7,1408,46.3,3,10.5
2,1,298.1,308.5,1498,49.4,5,10.4
3,1,298.2,308.6,1433,39.5,7,10.4
4,1,298.2,308.7,1408,40.0,9,10.5


In [6]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Machine failure, dtype: int64

# train test split

In [9]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.30,random_state=42)

In [21]:
for i in y_train, y_test:
    counts = pd.DataFrame(i).value_counts()
    print(counts)
# y_train -> is imbalanced - handled using SMOTE

Machine failure
0                  6754
1                   246
Name: count, dtype: int64
Machine failure
0                  2907
1                    93
Name: count, dtype: int64


# standardization

In [10]:
# using Robust Scaler --> Keeps the exact extreme values but scales the "normal" data around them.
from sklearn.preprocessing import RobustScaler
transformer = RobustScaler()
x_train_scaled= transformer.fit_transform(x_train)
x_train_scaled

array([[ 1.        , -0.928     , -0.82608696, ..., -0.88970588,
         0.24074074,  0.70588235],
       [ 1.        , -0.256     , -0.39130435, ...,  0.44852941,
        -0.7037037 ,  0.05882353],
       [ 1.        ,  0.128     ,  0.82608696, ...,  1.51470588,
         0.36111111,  1.        ],
       ...,
       [-1.        ,  0.864     ,  0.95652174, ...,  0.51470588,
         1.07407407, -0.17647059],
       [-1.        , -1.28      , -1.39130435, ..., -0.55882353,
        -0.68518519,  0.58823529],
       [ 0.        ,  0.032     ,  0.13043478, ..., -1.05882353,
        -0.66666667,  0.23529412]])

# handling imbalanced data - for training set

In [11]:
from imblearn.over_sampling import SMOTE    # need to be applied only in training data - to avoid data leakage
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_scaled, y_train)

In [13]:
y_train_resampled.value_counts()

Machine failure
0    6754
1    6754
Name: count, dtype: int64

# Model Creation

In [ ]:
# model prediction & evaluation 
def cm_pred_eval(classifier,x_test):
    # prediction
    y_pred=classifier.predict(x_test)
    # confusion matrix
    from sklearn.metrics import confusion_matrix
    cm=confusion_matrix(y_test,y_pred)
    # classification report 
    from sklearn.metrics import classification_report
    classi_report=classification_report(y_test,y_pred)
    # accuracy 
    from sklearn.metrics import accuracy_score
    accuracy=accuracy_score(y_test,y_pred)
    return classifier,accuracy,cm,classi_report,x_test,y_test

### --------------- classification algortihms -----------------------------
# logistic regression 
def logistic(x_train,y_train,x_test):
    # model creation
    from sklearn.linear_model import LogisticRegression
    classifier=LogisticRegression(random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# KNN 
def knn(x_train,y_train,x_test):
    # model creation
    from sklearn.neighbors import KNeighborsClassifier
    classifier=KNeighborsClassifier(n_neighbors=5,p=2, metric='minkowski')
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - linear 
def svm_linear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='linear',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# SVM - Non linear 
def svm_nonlinear(x_train,y_train,x_test):
    # model creation
    from sklearn.svm import SVC
    classifier=SVC(kernel='rbf',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# navie bayes
def naive(x_train,y_train,x_test):
    # model creation 
    from sklearn.naive_bayes import GaussianNB
    classifier=GaussianNB()
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# decision tree
def decision_tree(x_train,y_train,x_test):
     # model creation 
    from sklearn.tree import DecisionTreeClassifier
    classifier=DecisionTreeClassifier(criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

# random forest
def random_forest(x_train,y_train,x_test):
     # model creation 
    from sklearn.ensemble import RandomForestClassifier
    classifier=RandomForestClassifier(n_estimators=10,criterion='entropy',random_state=0)
    classifier.fit(x_train,y_train)
    # model prediction & evaluation
    classifier,accuracy,cm,classi_report,x_test,y_test = cm_pred_eval(classifier,x_test)
    return classifier,accuracy,cm,classi_report,x_test,y_test

def 